<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Limpieza_SQL_(Simple).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ----------------------------------------
# 1. IMPORTAR LIBRERÍAS Y CARGAR DATASET
# ----------------------------------------
import sqlite3
import pandas as pd
import seaborn as sns

# Cargar dataset Penguins
penguins = sns.load_dataset("penguins")

# Mostrar datos originales
display(penguins.head())

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [2]:
# ----------------------------------------
# 2. GUARDAR DATASET EN UNA BASE SQL
# ----------------------------------------
conn = sqlite3.connect(":memory:")
penguins.to_sql("penguins", conn, index=False, if_exists="replace")

print("Base de datos creada en memoria.\n")


Base de datos creada en memoria.



In [4]:
# ----------------------------------------
# 3. VERIFICAR NULOS CON SQL
# ----------------------------------------
print("\nCantidad de valores nulos por columna:")
display(pd.read_sql_query("""
SELECT
    SUM(CASE WHEN species IS NULL THEN 1 ELSE 0 END) AS species_null,
    SUM(CASE WHEN island IS NULL THEN 1 ELSE 0 END) AS island_null,
    SUM(CASE WHEN bill_length_mm IS NULL THEN 1 ELSE 0 END) AS bill_length_null,
    SUM(CASE WHEN bill_depth_mm IS NULL THEN 1 ELSE 0 END) AS bill_depth_null,
    SUM(CASE WHEN flipper_length_mm IS NULL THEN 1 ELSE 0 END) AS flipper_null,
    SUM(CASE WHEN body_mass_g IS NULL THEN 1 ELSE 0 END) AS body_mass_null,
    SUM(CASE WHEN sex IS NULL THEN 1 ELSE 0 END) AS sex_null
FROM penguins
""", conn))


Cantidad de valores nulos por columna:


,species_null,island_null,bill_length_null,bill_depth_null,flipper_null,body_mass_null,sex_null
0,0,0,2,2,2,2,11


In [5]:
# ----------------------------------------
# 4. CREAR UNA TABLA LIMPIA CON SQL
# ----------------------------------------
# Aquí limpiamos:
# - Nulos de valores numéricos con promedio
# - Nulos de valores categóricos con modo
# - Estandarizamos sexo a 0/1
# - Quitamos registros totalmente inválidos

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE penguins_clean AS
SELECT
    species,
    island,

    -- Rellenar nulos con promedio
    COALESCE(bill_length_mm, (SELECT AVG(bill_length_mm) FROM penguins)) AS bill_length_mm,
    COALESCE(bill_depth_mm, (SELECT AVG(bill_depth_mm) FROM penguins)) AS bill_depth_mm,
    COALESCE(flipper_length_mm, (SELECT AVG(flipper_length_mm) FROM penguins)) AS flipper_length_mm,
    COALESCE(body_mass_g, (SELECT AVG(body_mass_g) FROM penguins)) AS body_mass_g,

    -- Estandarización de sexo: male = 1, female = 0
    CASE
        WHEN sex = 'Male' THEN 1
        WHEN sex = 'Female' THEN 0
        ELSE 0  -- si venía NULL
    END AS sex_code

FROM penguins
WHERE species IS NOT NULL   -- elimina filas totalmente dañadas
""")

print("\nTabla limpia creada con éxito.\n")


Tabla limpia creada con éxito.



In [6]:
# ----------------------------------------
# 5. MOSTRAR RESULTADO LIMPIO
# ----------------------------------------
df_clean = pd.read_sql_query("SELECT * FROM penguins_clean", conn)
display(df_clean.head())


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex_code
0,Adelie,Torgersen,39.10000,18.70000,181.000000,3750.000000,1
1,Adelie,Torgersen,39.50000,17.40000,186.000000,3800.000000,0
2,Adelie,Torgersen,40.30000,18.00000,195.000000,3250.000000,0
3,Adelie,Torgersen,43.92193,17.15117,200.915205,4201.754386,0
4,Adelie,Torgersen,36.70000,19.30000,193.000000,3450.000000,0


In [8]:
# ----------------------------------------
# 6. ESTADÍSTICAS LIMPIAS CON SQL
# ----------------------------------------
print("\nPromedios por especie ya con datos limpios:")
display(pd.read_sql_query("""
SELECT
    species,
    AVG(bill_length_mm) AS bill_length_avg,
    AVG(flipper_length_mm) AS flipper_avg,
    AVG(body_mass_g) AS mass_avg
FROM penguins_clean
GROUP BY species
""", conn))


Promedios por especie ya con datos limpios:


,species,bill_length_avg,flipper_avg,mass_avg
0,Adelie,38.825144,190.025758,3703.958910
1,Chinstrap,48.833824,195.823529,3733.088235
2,Gentoo,47.475983,217.055768,5068.965761


In [10]:
# ----------------------------------------
# 7. VERIFICAR NULOS CON SQL
# ----------------------------------------
print("\nCantidad de valores nulos por columna:")
display(pd.read_sql_query("""
SELECT
    SUM(CASE WHEN species IS NULL THEN 1 ELSE 0 END) AS species_null,
    SUM(CASE WHEN island IS NULL THEN 1 ELSE 0 END) AS island_null,
    SUM(CASE WHEN bill_length_mm IS NULL THEN 1 ELSE 0 END) AS bill_length_null,
    SUM(CASE WHEN bill_depth_mm IS NULL THEN 1 ELSE 0 END) AS bill_depth_null,
    SUM(CASE WHEN flipper_length_mm IS NULL THEN 1 ELSE 0 END) AS flipper_null,
    SUM(CASE WHEN body_mass_g IS NULL THEN 1 ELSE 0 END) AS body_mass_null,
    SUM(CASE WHEN sex_code IS NULL THEN 1 ELSE 0 END) AS sex_null
FROM penguins_clean
""", conn))


Cantidad de valores nulos por columna:


,species_null,island_null,bill_length_null,bill_depth_null,flipper_null,body_mass_null,sex_null
0,0,0,0,0,0,0,0
